# 01 — SWaT Dataset Physical Architecture & Exploratory Data Analysis
**Course:** BCSE209P — Machine Learning Lab  
**Students:** P Naga Venkata Kushal (23BDS0133) & Koteru Tarun (23BDS0103)  
**Faculty Guide:** Dr. Dega Nagaraju  
**Repository:** [Reliability-Aware Anomaly Detection](https://github.com/nvk321/Reliability-Aware-Anomaly-Detection)

---
### Objectives
1. Inspect the physical architecture of the Secure Water Treatment (SWaT) testbed (Stages P1 through P6).
2. Explore the 51 sensor and actuator channels and their physical roles.
3. Examine the dataset splits: pure normal training data (7 days) vs. mixed test data with 36 cyber-attacks (4 days).
4. Analyze the label distribution (11.98% attack density) and the critical distinction between cyber-attacks and sensor faults.


In [ ]:
import os
import sys
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Add project root to sys.path
sys.path.insert(0, os.path.abspath('..'))

from src.visualization.plots import apply_scientific_style
apply_scientific_style()

print('Imports successful!')


### 1. Load Dataset Metadata and Split Summary
The dataset summary records the exact sample counts across the downsampled splits.


In [ ]:
summary_path = os.path.join('..', 'data', 'processed', 'SWaT', 'dataset_summary.json')
with open(summary_path, 'r') as f:
    dataset_summary = json.load(f)

names_path = os.path.join('..', 'data', 'processed', 'SWaT', 'sensor_names.json')
with open(names_path, 'r') as f:
    sensor_names = json.load(f)

print(f"Dataset: {dataset_summary['dataset']}")
print(f"Downsampling Rate: {dataset_summary['downsample_rate']}x (10s sampling interval)")
print(f"Total Sensor Channels: {len(sensor_names)}")
print(f"Training Samples (y=0, 100% Normal): {dataset_summary['train_samples']:,}")
print(f"Validation Samples (y=0, 100% Normal): {dataset_summary['val_samples']:,}")
print(f"Test Samples: {dataset_summary['test_samples']:,} (Attacks: {dataset_summary['test_anomalies']:,} = {dataset_summary['test_attack_rate']*100:.2f}%)")


### 2. Physical Subsystem Stage Breakdown
SWaT comprises 6 continuous physical stages:
- **P1 (Raw Water):** Flow meter `FIT101`, Level sensor `LIT101`, Inflow valve `MV101`, Pump `P101`/`P102`
- **P2 (Chemical Dosing):** pH meter `AIT201`, Conductivity `AIT202`/`AIT203`, Dosing pumps `P201`-`P206`
- **P3 (Ultrafiltration):** Differential pressure `DPIT301`, UF feed pumps `P301`/`P302`, Backwash valve `MV301`-`MV304`
- **P4 (De-chlorination):** ORP analyzer `AIT401`/`AIT402`, UV lamps `UV401`, Booster pump `P401`/`P402`
- **P5 (Reverse Osmosis):** RO feed pressure `AIT501`-`AIT504`, High pressure pumps `P501`/`P502`
- **P6 (Backwash Recycling):** Recycle pump `P601`/`P602`


In [ ]:
stages = {
    'P1 (Raw Water)': [s for s in sensor_names if s.startswith(('FIT1', 'LIT1', 'MV1', 'P1'))],
    'P2 (Chemical Dosing)': [s for s in sensor_names if s.startswith(('AIT2', 'FIT2', 'MV2', 'P2'))],
    'P3 (Ultrafiltration)': [s for s in sensor_names if s.startswith(('DPIT3', 'FIT3', 'LIT3', 'MV3', 'P3'))],
    'P4 (De-chlorination)': [s for s in sensor_names if s.startswith(('AIT4', 'FIT4', 'LIT4', 'P4', 'UV4'))],
    'P5 (Reverse Osmosis)': [s for s in sensor_names if s.startswith(('AIT5', 'FIT5', 'P5', 'MV5'))],
    'P6 (Recycling)': [s for s in sensor_names if s.startswith(('P6', 'FIT6', 'LIT6'))],
}

for stage, channels in stages.items():
    print(f"{stage:25s}: {len(channels):2d} channels -> {', '.join(channels[:5])}{'...' if len(channels) > 5 else ''}")


### 3. Load Sample Preprocessed Arrays & Inspect Sensor Telemetry
We load a small slice of the test array to visualize how continuous telemetry changes during an attack.


In [ ]:
data_dir = os.path.join('..', 'data', 'processed', 'SWaT')
test_data = np.load(os.path.join(data_dir, 'test.npy'))
test_labels = np.load(os.path.join(data_dir, 'test_labels.npy'))

# Slice around Attack Scenario 1 (Step 1,500: LIT101 Tank Inflow Overflow)
slice_start, slice_len = 1400, 300
sub_test = test_data[slice_start:slice_start + slice_len]
sub_labels = test_labels[slice_start:slice_start + slice_len]

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(12, 6), sharex=True)

# Plot LIT101 and FIT101
lit_idx = sensor_names.index('LIT101')
fit_idx = sensor_names.index('FIT101')
time_axis = np.arange(slice_start, slice_start + slice_len)

ax1.plot(time_axis, sub_test[:, lit_idx], label='LIT101 (Raw Tank Level)', color='#1D4ED8', lw=1.5)
ax1.plot(time_axis, sub_test[:, fit_idx], label='FIT101 (Inflow Rate)', color='#059669', lw=1.5)
ax1.axvspan(time_axis[sub_labels == 1][0], time_axis[sub_labels == 1][-1], color='#EF4444', alpha=0.25, label='Physical Attack Period')
ax1.set_ylabel('Standardized Value')
ax1.set_title('SWaT Attack Scenario 1: LIT101 Tank Overflow Manipulation', fontweight='bold')
ax1.legend(loc='upper right')

# Plot attack ground truth binary signal
ax2.fill_between(time_axis, sub_labels, color='#EF4444', step='mid', alpha=0.6, label='Attack Label (y=1)')
ax2.set_ylabel('Ground Truth')
ax2.set_xlabel('Time Step (10s intervals)')
ax2.set_ylim(-0.1, 1.2)
ax2.legend(loc='upper right')

plt.tight_layout()
plt.show()


### Viva Voce Key Takeaways — Section A: Data
1. **SWaT Characteristics:** 51 variables (26 physical sensors, 25 discrete actuators/valves/pumps).
2. **Attack Nature:** Attacks are physical disruptions (e.g. altering level thresholds to cause overflow, spoofing pH meters).
3. **Downsampling Rationale:** 10s downsampling captures physical transport dynamics (hydraulic tanks take minutes to fill) while removing high-frequency sensor noise and making deep learning computationally feasible on edge hardware.
